# Zoo Valence/Arousal Vision Prep

Prepare a compact OpenFARM visual affect eval from the Figshare/Nature zoo emotion-recognition study.

Goals:
- verify the public source files and labels;
- split the three video compilations into the 15 expert-labeled stimuli using black-frame separators;
- audit the source audio before deciding how to expose it;
- publish bounded visual media with opaque filenames;
- include both muted short videos and 3x3 filmstrips for video/image model ablations;
- optionally export an audio-bearing video variant for Gemma-style multimodal ablations;
- keep the Hugging Face rows source-shaped, not env-shaped: no pre-rendered messages, task ids, OpenFARM row ids, clip codes, timestamps, source filenames, or expert interpretation notes.

This is eval-only. There are only 15 expert-labeled clips, so a train split would be misleading.

In [1]:
#@title Setup dependencies and imports
!pip -q install datasets huggingface_hub opencv-python-headless pillow pandas requests

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re
import subprocess

import cv2
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont
from datasets import Dataset, DatasetDict, Features, Image as HFImage, Value, load_dataset
from huggingface_hub import HfApi, login

try:
    from datasets import Video as HFVideo
except Exception:
    HFVideo = None


In [2]:
#@title Configuration
FIGSHARE_COLLECTION_ID = 7807931
FIGSHARE_COLLECTION_API = f"https://api.figshare.com/v2/collections/{FIGSHARE_COLLECTION_ID}"
SOURCE_URL = "https://doi.org/10.6084/m9.figshare.c.7807931"
SOURCE_DOI = "10.6084/m9.figshare.c.7807931"
ARTICLE_URL = "https://www.nature.com/articles/s41598-025-28646-7"
ARTICLE_DOI = "10.1038/s41598-025-28646-7"

REQUIRED_SOURCE_FILES = {
    "Video_clips2022_1.m4v": "8340964d6990cd8c09c8a32da27e35c4",
    "Video_clips2022_2.m4v": "ccacaea1095eb9a11f67264018b2caf2",
    "Video_clips2022_3.m4v": "567306eac6b4736f78bbe67ceda16e5f",
    "Emotions of the video clips 2022.pdf": "333c0867f1d8e3fc0987b610bef49031",
}

HF_REPO_ID = "oliveirabruno01/openfarm-zoo-valence-arousal"  #@param {type:"string"}
PUSH_TO_HUB = True  #@param {type:"boolean"}
PRIVATE_REPO = False  #@param {type:"boolean"}

# Primary benchmark media are muted because the source paper frames the task as
# visual emotion recognition from mute video clips. Flip this only for an
# explicit Gemma 4 multimodal ablation; do not treat it as the headline task.
EXPORT_AUDIO_VARIANT = False  #@param {type:"boolean"}
REQUIRE_VIDEO_FOR_PUSH = True  #@param {type:"boolean"}
REQUIRE_FILMSTRIPS_FOR_PUSH = True  #@param {type:"boolean"}

VIDEO_WIDTH = 854  #@param {type:"integer"}
VIDEO_CRF = 23  #@param {type:"integer"}
FILMSTRIP_FRAMES = 9  #@param {type:"integer"}
FILMSTRIP_THUMB_WIDTH = 426  #@param {type:"integer"}
FILMSTRIP_THUMB_HEIGHT = 240  #@param {type:"integer"}
AUDIO_SILENCE_DB = -45  #@param {type:"integer"}

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").exists() and (p / "data").exists()), Path.cwd())
probe_dir = repo_root / "data" / "zoo_valence_arousal_probe"
default_work_dir = probe_dir / "prepared" if probe_dir.exists() else Path("/content/openfarm_zoo_valence_arousal")
default_raw_dir = probe_dir / "raw" if (probe_dir / "raw").exists() else default_work_dir / "raw"

WORK_DIR = Path(os.environ.get("OPENFARM_ZOO_WORK_DIR", str(default_work_dir))).expanduser()
RAW_DIR = Path(os.environ.get("OPENFARM_ZOO_RAW_DIR", str(default_raw_dir))).expanduser()
if not WORK_DIR.is_absolute():
    WORK_DIR = (repo_root / WORK_DIR).resolve()
if not RAW_DIR.is_absolute():
    RAW_DIR = (repo_root / RAW_DIR).resolve()

OUT_DIR = WORK_DIR
VIDEO_DIR = OUT_DIR / "video"
VIDEO_AUDIO_DIR = OUT_DIR / "video_with_audio"
FILMSTRIP_DIR = OUT_DIR / "filmstrip"
for d in [RAW_DIR, OUT_DIR, VIDEO_DIR, FILMSTRIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
if EXPORT_AUDIO_VARIANT:
    VIDEO_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("repo root:", repo_root)
print("raw dir:", RAW_DIR)
print("outputs:", OUT_DIR)
print("export audio variant:", EXPORT_AUDIO_VARIANT)

repo root: /content
raw dir: /content/openfarm_zoo_valence_arousal/raw
outputs: /content/openfarm_zoo_valence_arousal
export audio variant: False


In [3]:
#@title Download and verify Figshare source files

def md5_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_figshare_file_index() -> dict:
    articles = requests.get(f"{FIGSHARE_COLLECTION_API}/articles", timeout=60).json()
    detailed_articles = []
    files = {}
    for article in articles:
        detail = requests.get(f"https://api.figshare.com/v2/articles/{article['id']}", timeout=60).json()
        detailed_articles.append(detail)
        for file_info in detail.get("files", []):
            files[file_info["name"]] = file_info
    (RAW_DIR / "figshare_articles.json").write_text(json.dumps(detailed_articles, indent=2), encoding="utf-8")
    return files

missing = [name for name in REQUIRED_SOURCE_FILES if not (RAW_DIR / name).exists()]
if missing:
    print("downloading missing source files:", missing)
    file_index = fetch_figshare_file_index()
    for name in missing:
        if name not in file_index:
            raise FileNotFoundError(f"Figshare file not found in collection: {name}")
        out_path = RAW_DIR / name
        tmp_path = out_path.with_suffix(out_path.suffix + ".part")
        tmp_path.unlink(missing_ok=True)
        with requests.get(file_index[name]["download_url"], stream=True, timeout=120) as response:
            response.raise_for_status()
            with tmp_path.open("wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        tmp_path.rename(out_path)
else:
    print("all required source files already present")

for name, expected_md5 in REQUIRED_SOURCE_FILES.items():
    path = RAW_DIR / name
    actual_md5 = md5_file(path)
    assert actual_md5 == expected_md5, f"MD5 mismatch for {name}: {actual_md5} != {expected_md5}"
    print(name, path.stat().st_size, actual_md5)

downloading missing source files: ['Video_clips2022_1.m4v', 'Video_clips2022_2.m4v', 'Video_clips2022_3.m4v', 'Emotions of the video clips 2022.pdf']
Video_clips2022_1.m4v 80133232 8340964d6990cd8c09c8a32da27e35c4
Video_clips2022_2.m4v 72339805 ccacaea1095eb9a11f67264018b2caf2
Video_clips2022_3.m4v 74189650 567306eac6b4736f78bbe67ceda16e5f
Emotions of the video clips 2022.pdf 348071 333c0867f1d8e3fc0987b610bef49031


In [4]:
#@title Audited expert label table
# The source label PDF is human-readable, not a tidy machine table. These labels
# were audited against the PDF and the local probe metadata before building media.
CLIP_LABELS = [
    {"compilation": 1, "position": 1, "clip_code": "T3-", "species": "Siberian tiger", "valence": "negative", "arousal": "high", "expert_interpretation_note": "worried/slightly afraid facial expression"},
    {"compilation": 1, "position": 2, "clip_code": "M1n", "species": "Turkmenian markhor", "valence": "neutral", "arousal": "low", "expert_interpretation_note": "resting but observing"},
    {"compilation": 1, "position": 3, "clip_code": "B1+", "species": "Barbary macaque", "valence": "positive", "arousal": "low", "expert_interpretation_note": "receiving grooming, relaxed"},
    {"compilation": 1, "position": 4, "clip_code": "M4+", "species": "Turkmenian markhor", "valence": "positive", "arousal": "high", "expert_interpretation_note": "observing keeper bringing fresh grass"},
    {"compilation": 1, "position": 5, "clip_code": "M5-", "species": "Turkmenian markhor", "valence": "negative", "arousal": "high", "expert_interpretation_note": "observing/avoiding approaching machine"},
    {"compilation": 2, "position": 1, "clip_code": "B4-", "species": "Barbary macaque", "valence": "negative", "arousal": "high", "expert_interpretation_note": "dominance/threat signal"},
    {"compilation": 2, "position": 2, "clip_code": "M3-", "species": "Turkmenian markhor", "valence": "negative", "arousal": "high", "expert_interpretation_note": "mild threat over fresh grass"},
    {"compilation": 2, "position": 3, "clip_code": "T1n", "species": "Siberian tiger", "valence": "neutral", "arousal": "low", "expert_interpretation_note": "resting but observing"},
    {"compilation": 2, "position": 4, "clip_code": "B3+", "species": "Barbary macaque", "valence": "positive", "arousal": "high", "expert_interpretation_note": "active enrichment/foraging"},
    {"compilation": 2, "position": 5, "clip_code": "T4+", "species": "Siberian tiger", "valence": "positive", "arousal": "high", "expert_interpretation_note": "focused on carcass enrichment"},
    {"compilation": 3, "position": 1, "clip_code": "B5-", "species": "Barbary macaque", "valence": "negative", "arousal": "high", "expert_interpretation_note": "fear/submission signal"},
    {"compilation": 3, "position": 2, "clip_code": "B2n", "species": "Barbary macaque", "valence": "neutral", "arousal": "low", "expert_interpretation_note": "calm attentive sitting"},
    {"compilation": 3, "position": 3, "clip_code": "M2+", "species": "Turkmenian markhor", "valence": "positive", "arousal": "low", "expert_interpretation_note": "ruminating, relaxed"},
    {"compilation": 3, "position": 4, "clip_code": "T2+", "species": "Siberian tiger", "valence": "positive", "arousal": "low", "expert_interpretation_note": "grooming after eating"},
    {"compilation": 3, "position": 5, "clip_code": "T5-", "species": "Siberian tiger", "valence": "negative", "arousal": "high", "expert_interpretation_note": "fear/aggression toward another tiger"},
]
label_df = pd.DataFrame(CLIP_LABELS)

probe_metadata_path = probe_dir / "clips_metadata.csv"
if probe_metadata_path.exists():
    probe_df = pd.read_csv(probe_metadata_path)
    probe_cols = ["compilation", "position", "clip_code", "species", "valence", "arousal"]
    merged = label_df[probe_cols].merge(probe_df[probe_cols], on=probe_cols, how="outer", indicator=True)
    assert (merged["_merge"] == "both").all(), merged[merged["_merge"] != "both"]
    print("label table matches audited local probe metadata:", probe_metadata_path)
else:
    print("local probe metadata is not present; using the audited label table embedded in this notebook")

print("label rows:", len(label_df))
print("species:", dict(Counter(label_df["species"])))
print("valence:", dict(Counter(label_df["valence"])))
print("arousal:", dict(Counter(label_df["arousal"])))
label_df.sort_values(["compilation", "position"])[["compilation", "position", "clip_code", "species", "valence", "arousal", "expert_interpretation_note"]]

local probe metadata is not present; using the audited label table embedded in this notebook
label rows: 15
species: {'Siberian tiger': 5, 'Turkmenian markhor': 5, 'Barbary macaque': 5}
valence: {'negative': 6, 'neutral': 3, 'positive': 6}
arousal: {'high': 9, 'low': 6}


,compilation,position,clip_code,species,valence,arousal,expert_interpretation_note
0,1,1,T3-,Siberian tiger,negative,high,worried/slightly afraid facial expression
1,1,2,M1n,Turkmenian markhor,neutral,low,resting but observing
2,1,3,B1+,Barbary macaque,positive,low,"receiving grooming, relaxed"
3,1,4,M4+,Turkmenian markhor,positive,high,observing keeper bringing fresh grass
4,1,5,M5-,Turkmenian markhor,negative,high,observing/avoiding approaching machine
5,2,1,B4-,Barbary macaque,negative,high,dominance/threat signal
6,2,2,M3-,Turkmenian markhor,negative,high,mild threat over fresh grass
7,2,3,T1n,Siberian tiger,neutral,low,resting but observing
8,2,4,B3+,Barbary macaque,positive,high,active enrichment/foraging
9,2,5,T4+,Siberian tiger,positive,high,focused on carcass enrichment


In [5]:
#@title Detect black separators and derive stimulus segments
BLACK_RE = re.compile(r"black_start:(?P<start>[0-9.]+)\s+black_end:(?P<end>[0-9.]+)\s+black_duration:(?P<duration>[0-9.]+)")


def ffprobe_duration(path: Path) -> float:
    result = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path),
    ], check=True, capture_output=True, text=True)
    return float(result.stdout.strip())


def ffprobe_streams(path: Path) -> dict:
    result = subprocess.run([
        "ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path),
    ], check=True, capture_output=True, text=True)
    return json.loads(result.stdout)


def detect_black_segments(path: Path) -> list[dict]:
    result = subprocess.run([
        "ffmpeg", "-hide_banner", "-nostats", "-i", str(path),
        "-vf", "blackdetect=d=0.2:pix_th=0.10", "-an", "-f", "null", "-",
    ], capture_output=True, text=True, check=False)
    text = result.stderr + "\n" + result.stdout
    rows = []
    for match in BLACK_RE.finditer(text):
        rows.append({
            "source_video_filename": path.name,
            "start": round(float(match.group("start")), 3),
            "end": round(float(match.group("end")), 3),
            "duration": round(float(match.group("duration")), 3),
        })
    if len(rows) < 5:
        raise RuntimeError(f"Expected black separators in {path}, found {len(rows)}")
    return rows

source_files = {idx: RAW_DIR / f"Video_clips2022_{idx}.m4v" for idx in [1, 2, 3]}
black_rows = []
segment_rows = []
stream_rows = []
for compilation, source_path in source_files.items():
    stream_info = ffprobe_streams(source_path)
    for stream in stream_info.get("streams", []):
        stream_rows.append({
            "source_video_filename": source_path.name,
            "codec_type": stream.get("codec_type"),
            "codec_name": stream.get("codec_name"),
            "width": stream.get("width"),
            "height": stream.get("height"),
            "sample_rate": stream.get("sample_rate"),
            "channels": stream.get("channels"),
            "bit_rate": stream.get("bit_rate"),
            "duration": stream.get("duration"),
        })

    detected = detect_black_segments(source_path)
    black_rows.extend(detected)
    long_black = [row for row in detected if row["duration"] >= 2.5]
    assert len(long_black) == 5, f"{source_path.name}: expected 5 long separators, got {len(long_black)}"

    starts = [0.0] + [row["end"] for row in long_black[:4]]
    ends = [row["start"] for row in long_black[:5]]
    source_duration = ffprobe_duration(source_path)
    labels = label_df[label_df["compilation"] == compilation].sort_values("position").to_dict("records")
    for label, start, end in zip(labels, starts, ends):
        row = dict(label)
        row.update({
            "source_video_filename": source_path.name,
            "segment_start_sec": round(start, 3),
            "segment_end_sec": round(end, 3),
            "segment_duration_sec": round(end - start, 3),
            "source_video_duration_sec": round(source_duration, 3),
            "opaque_stem": f"zoo_clip_{len(segment_rows) + 1:04d}",
            "valence_arousal": f"{label['valence']}_{label['arousal']}",
        })
        segment_rows.append(row)

black_df = pd.DataFrame(black_rows)
segment_df = pd.DataFrame(segment_rows)
stream_df = pd.DataFrame(stream_rows)
black_df.to_csv(OUT_DIR / "black_segments_audit.csv", index=False)
segment_df.to_csv(OUT_DIR / "clip_segments_metadata.csv", index=False)
stream_df.to_csv(OUT_DIR / "source_streams_audit.csv", index=False)

assert len(segment_df) == 15
print("black separator rows:", len(black_df))
print("derived stimulus segments:", len(segment_df))
display(stream_df)
segment_df[["compilation", "position", "clip_code", "species", "valence", "arousal", "segment_start_sec", "segment_end_sec", "segment_duration_sec", "opaque_stem"]]

black separator rows: 45
derived stimulus segments: 15


,source_video_filename,codec_type,codec_name,width,height,sample_rate,channels,bit_rate,duration
0,Video_clips2022_1.m4v,video,h264,1280.0,720.0,None,NaN,3186176,194.366667
1,Video_clips2022_1.m4v,audio,aac,NaN,NaN,48000,2.0,105162,194.261313
2,Video_clips2022_2.m4v,video,h264,1280.0,720.0,None,NaN,3072809,181.266667
3,Video_clips2022_2.m4v,audio,aac,NaN,NaN,48000,2.0,112943,181.162646
4,Video_clips2022_3.m4v,video,h264,1280.0,720.0,None,NaN,3051734,186.633333
5,Video_clips2022_3.m4v,audio,aac,NaN,NaN,48000,2.0,121523,186.517313


,compilation,position,clip_code,species,valence,arousal,segment_start_sec,segment_end_sec,segment_duration_sec,opaque_stem
0,1,1,T3-,Siberian tiger,negative,high,0.000,33.133,33.133,zoo_clip_0001
1,1,2,M1n,Turkmenian markhor,neutral,low,36.133,70.933,34.800,zoo_clip_0002
2,1,3,B1+,Barbary macaque,positive,low,73.933,114.700,40.767,zoo_clip_0003
3,1,4,M4+,Turkmenian markhor,positive,high,117.700,150.200,32.500,zoo_clip_0004
4,1,5,M5-,Turkmenian markhor,negative,high,153.200,184.367,31.167,zoo_clip_0005
5,2,1,B4-,Barbary macaque,negative,high,0.000,38.567,38.567,zoo_clip_0006
6,2,2,M3-,Turkmenian markhor,negative,high,41.567,65.433,23.866,zoo_clip_0007
7,2,3,T1n,Siberian tiger,neutral,low,68.433,102.467,34.034,zoo_clip_0008
8,2,4,B3+,Barbary macaque,positive,high,105.467,137.367,31.900,zoo_clip_0009
9,2,5,T4+,Siberian tiger,positive,high,140.367,171.267,30.900,zoo_clip_0010


In [6]:
#@title Source audio audit
# The source files contain AAC audio, but the paper describes the study as visual
# recognition from mute clips. This audit tells us whether audio exists and how
# much of each derived stimulus is actually non-silent before we decide whether
# to export an explicit audio-bearing ablation variant.

def parse_silence_seconds(text: str, duration_sec: float) -> tuple[float, list[tuple[float, float, float]]]:
    current_start = None
    silence_sec = 0.0
    intervals = []
    for line in text.splitlines():
        if "silence_start:" in line:
            match = re.search(r"silence_start: ([0-9.]+)", line)
            if match:
                current_start = float(match.group(1))
        if "silence_end:" in line:
            match = re.search(r"silence_end: ([0-9.]+) \| silence_duration: ([0-9.]+)", line)
            if match:
                end = float(match.group(1))
                span = float(match.group(2))
                start = current_start if current_start is not None else end - span
                silence_sec += span
                intervals.append((round(start, 2), round(end, 2), round(span, 2)))
                current_start = None
    if current_start is not None:
        span = max(0.0, duration_sec - current_start)
        silence_sec += span
        intervals.append((round(current_start, 2), round(duration_sec, 2), round(span, 2)))
    return min(silence_sec, duration_sec), intervals


def source_volume_stats(path: Path) -> dict:
    result = subprocess.run([
        "ffmpeg", "-hide_banner", "-nostats", "-i", str(path), "-vn", "-af", "volumedetect", "-f", "null", "-",
    ], capture_output=True, text=True, check=False)
    text = result.stderr + result.stdout
    stats = {"source_video_filename": path.name}
    for key in ["mean_volume", "max_volume"]:
        match = re.search(rf"{key}:\s+(-?[0-9.]+) dB", text)
        stats[key + "_db"] = float(match.group(1)) if match else None
    return stats

volume_df = pd.DataFrame([source_volume_stats(path) for path in source_files.values()])
audio_rows = []
for row in segment_df.to_dict("records"):
    source_path = RAW_DIR / row["source_video_filename"]
    duration_sec = float(row["segment_duration_sec"])
    result = subprocess.run([
        "ffmpeg", "-hide_banner", "-nostats", "-ss", str(row["segment_start_sec"]), "-i", str(source_path),
        "-t", str(duration_sec), "-vn", "-af", f"silencedetect=n={AUDIO_SILENCE_DB}dB:d=0.25", "-f", "null", "-",
    ], capture_output=True, text=True, check=False)
    silence_sec, intervals = parse_silence_seconds(result.stderr + result.stdout, duration_sec)
    active_sec = max(0.0, duration_sec - silence_sec)
    audio_rows.append({
        "opaque_stem": row["opaque_stem"],
        "species": row["species"],
        "valence": row["valence"],
        "arousal": row["arousal"],
        "duration_sec": round(duration_sec, 3),
        "silence_sec": round(silence_sec, 2),
        "active_audio_sec": round(active_sec, 2),
        "active_audio_ratio": round(active_sec / duration_sec, 3),
        "first_silence_intervals": intervals[:5],
    })

audio_audit_df = pd.DataFrame(audio_rows)
audio_audit_df.to_csv(OUT_DIR / "audio_presence_audit.csv", index=False)
print("source volume stats")
display(volume_df)
print("derived segment audio presence")
display(audio_audit_df[["opaque_stem", "species", "valence", "arousal", "duration_sec", "active_audio_sec", "active_audio_ratio"]])
print(audio_audit_df[["active_audio_sec", "active_audio_ratio"]].describe())

source volume stats


,source_video_filename,mean_volume_db,max_volume_db
0,Video_clips2022_1.m4v,-33.1,-4.1
1,Video_clips2022_2.m4v,-37.5,-12.5
2,Video_clips2022_3.m4v,-30.5,-5.4


derived segment audio presence


,opaque_stem,species,valence,arousal,duration_sec,active_audio_sec,active_audio_ratio
0,zoo_clip_0001,Siberian tiger,negative,high,33.133,9.62,0.290
1,zoo_clip_0002,Turkmenian markhor,neutral,low,34.800,14.80,0.425
2,zoo_clip_0003,Barbary macaque,positive,low,40.767,22.13,0.543
3,zoo_clip_0004,Turkmenian markhor,positive,high,32.500,10.01,0.308
4,zoo_clip_0005,Turkmenian markhor,negative,high,31.167,15.92,0.511
5,zoo_clip_0006,Barbary macaque,negative,high,38.567,15.42,0.400
6,zoo_clip_0007,Turkmenian markhor,negative,high,23.866,8.02,0.336
7,zoo_clip_0008,Siberian tiger,neutral,low,34.034,13.75,0.404
8,zoo_clip_0009,Barbary macaque,positive,high,31.900,15.33,0.481
9,zoo_clip_0010,Siberian tiger,positive,high,30.900,17.84,0.577


       active_audio_sec  active_audio_ratio
count         15.000000           15.000000
mean          15.034667            0.450333
std            4.215202            0.094906
min            8.020000            0.290000
25%           11.980000            0.383000
50%           15.420000            0.481000
75%           17.030000            0.527000
max           22.130000            0.577000


In [7]:
#@title Extract bounded media with opaque filenames
# Primary videos are muted to match the source study framing. If EXPORT_AUDIO_VARIANT=True,
# a second video_with_audio column is added for explicit multimodal ablations.

def extract_video(source_path: Path, start_sec: float, duration_sec: float, out_path: Path, keep_audio: bool):
    cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-ss", f"{start_sec:.3f}", "-i", str(source_path), "-t", f"{duration_sec:.3f}",
    ]
    if keep_audio:
        cmd.extend(["-map", "0:v:0", "-map", "0:a:0?", "-vf", f"scale={VIDEO_WIDTH}:-2", "-c:v", "libx264", "-preset", "veryfast", "-crf", str(VIDEO_CRF), "-c:a", "aac", "-b:a", "128k"])
    else:
        cmd.extend(["-an", "-vf", f"scale={VIDEO_WIDTH}:-2", "-c:v", "libx264", "-preset", "veryfast", "-crf", str(VIDEO_CRF)])
    cmd.extend(["-movflags", "+faststart", str(out_path)])
    subprocess.run(cmd, check=True)


def make_filmstrip(video_path: Path, out_path: Path, n_frames: int = FILMSTRIP_FRAMES, thumb_size: tuple[int, int] = (FILMSTRIP_THUMB_WIDTH, FILMSTRIP_THUMB_HEIGHT)) -> list[float]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = frame_count / fps if frame_count else ffprobe_duration(video_path)
    sample_times = np.linspace(max(0.05, duration * 0.05), max(0.06, duration * 0.95), n_frames)

    frames = []
    used_times = []
    for sample_time in sample_times:
        cap.set(cv2.CAP_PROP_POS_MSEC, float(sample_time) * 1000)
        ok, frame = cap.read()
        if not ok:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, frame_count - 1))
            ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"Could not sample frame at {sample_time:.3f}s from {video_path}")
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(rgb).resize(thumb_size, Image.Resampling.LANCZOS))
        used_times.append(round(float(sample_time), 3))
    cap.release()

    grid = int(np.ceil(np.sqrt(n_frames)))
    canvas = Image.new("RGB", (thumb_size[0] * grid, thumb_size[1] * grid), "white")
    for idx, frame in enumerate(frames):
        canvas.paste(frame, ((idx % grid) * thumb_size[0], (idx // grid) * thumb_size[1]))
    canvas.save(out_path, quality=88, optimize=True)
    return used_times

for path in list(VIDEO_DIR.glob("*.mp4")) + list(FILMSTRIP_DIR.glob("*.jpg")):
    path.unlink()
if EXPORT_AUDIO_VARIANT:
    for path in VIDEO_AUDIO_DIR.glob("*.mp4"):
        path.unlink()

sampled_frames = []
public_rows = []
for row in segment_df.sort_values(["compilation", "position"]).to_dict("records"):
    source_path = RAW_DIR / row["source_video_filename"]
    video_path = VIDEO_DIR / f"{row['opaque_stem']}.mp4"
    filmstrip_path = FILMSTRIP_DIR / f"{row['opaque_stem']}_filmstrip.jpg"

    extract_video(source_path, row["segment_start_sec"], row["segment_duration_sec"], video_path, keep_audio=False)
    sample_times = make_filmstrip(video_path, filmstrip_path)
    sampled_frames.append({"video_filename": video_path.name, "sample_times_sec": sample_times})

    public_row = {
        "species": row["species"],
        "valence": row["valence"],
        "arousal": row["arousal"],
        "valence_arousal": row["valence_arousal"],
        "video_filename": video_path.name,
        "filmstrip_filename": filmstrip_path.name,
        "video_sha256": sha256_file(video_path),
        "filmstrip_sha256": sha256_file(filmstrip_path),
        "duration_sec": round(ffprobe_duration(video_path), 3),
        "source_url": SOURCE_URL,
        "source_article_url": ARTICLE_URL,
        "source_doi": SOURCE_DOI,
        "article_doi": ARTICLE_DOI,
        "license": "cc-by-4.0",
        "split": "test",
        "video": str(video_path),
        "filmstrip": str(filmstrip_path),
    }

    if EXPORT_AUDIO_VARIANT:
        audio_video_path = VIDEO_AUDIO_DIR / f"{row['opaque_stem']}_with_audio.mp4"
        extract_video(source_path, row["segment_start_sec"], row["segment_duration_sec"], audio_video_path, keep_audio=True)
        public_row.update({
            "video_with_audio_filename": audio_video_path.name,
            "video_with_audio_sha256": sha256_file(audio_video_path),
            "video_with_audio": str(audio_video_path),
        })

    public_rows.append(public_row)

public_df = pd.DataFrame(public_rows)
(OUT_DIR / "sampled_filmstrip_frames.json").write_text(json.dumps(sampled_frames, indent=2), encoding="utf-8")
with (OUT_DIR / "zoo_valence_arousal_hf_dataset.jsonl").open("w", encoding="utf-8") as f:
    for row in public_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

assert len(public_df) == 15
assert public_df["video_filename"].str.contains("neg|pos|positive|negative|neutral|high|low", case=False).sum() == 0
if EXPORT_AUDIO_VARIANT:
    assert public_df["video_with_audio_filename"].str.contains("neg|pos|positive|negative|neutral|high|low", case=False).sum() == 0

public_df.head()[["video_filename", "filmstrip_filename", "species", "valence", "arousal", "duration_sec"]]

,video_filename,filmstrip_filename,species,valence,arousal,duration_sec
0,zoo_clip_0001.mp4,zoo_clip_0001_filmstrip.jpg,Siberian tiger,negative,high,33.134
1,zoo_clip_0002.mp4,zoo_clip_0002_filmstrip.jpg,Turkmenian markhor,neutral,low,34.800
2,zoo_clip_0003.mp4,zoo_clip_0003_filmstrip.jpg,Barbary macaque,positive,low,40.800
3,zoo_clip_0004.mp4,zoo_clip_0004_filmstrip.jpg,Turkmenian markhor,positive,high,32.500
4,zoo_clip_0005.mp4,zoo_clip_0005_filmstrip.jpg,Turkmenian markhor,negative,high,31.200


In [8]:
#@title Strategic data shape and visual audit

def draw_bar_chart(path: Path, rows: list[dict]):
    width, height = 1200, 520
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((30, 24), "Zoo Valence/Arousal Eval Shape (test-only)", fill="black", font=font)
    panels = [
        ("Species", Counter(row["species"] for row in rows)),
        ("Valence", Counter(row["valence"] for row in rows)),
        ("Arousal", Counter(row["arousal"] for row in rows)),
    ]
    palette = [(77, 119, 190), (216, 122, 79), (91, 160, 111), (130, 95, 170)]
    for panel_idx, (name, counts) in enumerate(panels):
        x0 = 35 + panel_idx * 385
        y0 = 80
        draw.text((x0, y0), name, fill="black", font=font)
        max_count = max(counts.values())
        for idx, (label, count) in enumerate(counts.items()):
            y = y0 + 45 + idx * 75
            bar_width = int(210 * count / max_count)
            draw.rectangle([x0 + 130, y, x0 + 130 + bar_width, y + 30], fill=palette[idx % len(palette)])
            draw.text((x0, y + 8), str(label), fill="black", font=font)
            draw.text((x0 + 140 + bar_width, y + 8), str(count), fill="black", font=font)
    image.save(path, quality=95)

summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "rows": len(public_df),
    "split_counts": dict(Counter(public_df["split"])),
    "species_counts": dict(Counter(public_df["species"])),
    "valence_counts": dict(Counter(public_df["valence"])),
    "arousal_counts": dict(Counter(public_df["arousal"])),
    "valence_arousal_counts": dict(Counter(public_df["valence_arousal"])),
    "duration_sec": {
        "min": float(public_df["duration_sec"].min()),
        "median": float(public_df["duration_sec"].median()),
        "max": float(public_df["duration_sec"].max()),
        "total_minutes": float(public_df["duration_sec"].sum() / 60),
    },
    "video_total_mib": round(sum((VIDEO_DIR / name).stat().st_size for name in public_df["video_filename"]) / 1024 / 1024, 2),
    "filmstrip_total_mib": round(sum((FILMSTRIP_DIR / name).stat().st_size for name in public_df["filmstrip_filename"]) / 1024 / 1024, 2),
    "audio_variant_exported": bool(EXPORT_AUDIO_VARIANT),
    "source_audio_note": "Source compilations contain AAC stereo audio, but the paper frames the task as mute visual recognition; audio-bearing video should be treated as an explicit ablation only.",
    "segmentation_policy": "Segments are derived from long black separators in the Figshare compilations; each row preserves the full human-study stimulus segment: real-speed clip, slow-motion clip, and still-frame portion, excluding inter-example separator/end black screens.",
    "public_schema_policy": "Public rows use opaque media filenames and do not include clip_code, source filenames, segment timestamps, expert interpretation notes, messages, task ids, or OpenFARM-specific row ids.",
}
if EXPORT_AUDIO_VARIANT:
    summary["video_with_audio_total_mib"] = round(sum((VIDEO_AUDIO_DIR / name).stat().st_size for name in public_df["video_with_audio_filename"]) / 1024 / 1024, 2)

(OUT_DIR / "dataset_audit.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
draw_bar_chart(OUT_DIR / "zoo_split_label_audit.png", public_rows)

print(json.dumps(summary, indent=2))
public_df.groupby(["valence", "arousal", "species"]).size().rename("n").reset_index()

{
  "generated_at": "2026-05-18T05:06:00.991095+00:00",
  "rows": 15,
  "split_counts": {
    "test": 15
  },
  "species_counts": {
    "Siberian tiger": 5,
    "Turkmenian markhor": 5,
    "Barbary macaque": 5
  },
  "valence_counts": {
    "negative": 6,
    "neutral": 3,
    "positive": 6
  },
  "arousal_counts": {
    "high": 9,
    "low": 6
  },
  "valence_arousal_counts": {
    "negative_high": 6,
    "neutral_low": 3,
    "positive_low": 3,
    "positive_high": 3
  },
  "duration_sec": {
    "min": 23.867,
    "median": 32.5,
    "max": 40.8,
    "total_minutes": 8.273916666666665
  },
  "video_total_mib": 46.78,
  "filmstrip_total_mib": 4.22,
  "audio_variant_exported": false,
  "source_audio_note": "Source compilations contain AAC stereo audio, but the paper frames the task as mute visual recognition; audio-bearing video should be treated as an explicit ablation only.",
  "segmentation_policy": "Segments are derived from long black separators in the Figshare compilations; each

,valence,arousal,species,n
0,negative,high,Barbary macaque,2
1,negative,high,Siberian tiger,2
2,negative,high,Turkmenian markhor,2
3,neutral,low,Barbary macaque,1
4,neutral,low,Siberian tiger,1
5,neutral,low,Turkmenian markhor,1
6,positive,high,Barbary macaque,1
7,positive,high,Siberian tiger,1
8,positive,high,Turkmenian markhor,1
9,positive,low,Barbary macaque,1


In [9]:
#@title Validate leakage guards and build Hugging Face DatasetDict preview
PUBLIC_FORBIDDEN_COLUMNS = {
    "clip_code",
    "source_video_filename",
    "segment_start_sec",
    "segment_end_sec",
    "segment_duration_sec",
    "expert_interpretation_note",
    "openfarm_id",
    "task_id",
    "messages",
}
assert not (PUBLIC_FORBIDDEN_COLUMNS & set(public_df.columns)), PUBLIC_FORBIDDEN_COLUMNS & set(public_df.columns)
assert not any(token in json.dumps(public_rows) for token in ["openfarm_id", "task_id", "messages"])
if HFVideo is None:
    raise RuntimeError("datasets.Video is unavailable; update datasets before building the publishable video dataset.")

base_feature_spec = {
    "species": Value("string"),
    "valence": Value("string"),
    "arousal": Value("string"),
    "valence_arousal": Value("string"),
    "video_filename": Value("string"),
    "filmstrip_filename": Value("string"),
    "video_sha256": Value("string"),
    "filmstrip_sha256": Value("string"),
    "duration_sec": Value("float64"),
    "source_url": Value("string"),
    "source_article_url": Value("string"),
    "source_doi": Value("string"),
    "article_doi": Value("string"),
    "license": Value("string"),
    "video": HFVideo(decode=False),
    "filmstrip": HFImage(decode=True),
}
if EXPORT_AUDIO_VARIANT:
    base_feature_spec.update({
        "video_with_audio_filename": Value("string"),
        "video_with_audio_sha256": Value("string"),
        "video_with_audio": HFVideo(decode=False),
    })


def media_record(local_path_value, opaque_filename: str, expected_sha256: str) -> dict:
    local_path = Path(local_path_value)
    if not local_path.exists():
        raise FileNotFoundError(f"Missing media file: {local_path}")
    raw = local_path.read_bytes()
    actual_sha256 = hashlib.sha256(raw).hexdigest()
    assert actual_sha256 == expected_sha256, f"SHA256 mismatch for {local_path.name}: {actual_sha256} != {expected_sha256}"
    return {"path": opaque_filename, "bytes": raw}


def build_publish_records(frame: pd.DataFrame) -> list[dict]:
    records = frame.to_dict("records")
    for record in records:
        record["video"] = media_record(record["video"], record["video_filename"], record["video_sha256"])
        record["filmstrip"] = media_record(record["filmstrip"], record["filmstrip_filename"], record["filmstrip_sha256"])
        if EXPORT_AUDIO_VARIANT:
            record["video_with_audio"] = media_record(
                record["video_with_audio"],
                record["video_with_audio_filename"],
                record["video_with_audio_sha256"],
            )
    return records

split_datasets = {}
for split_name in ["test"]:
    frame = public_df[public_df["split"] == split_name].drop(columns=["split"]).reset_index(drop=True)
    split_datasets[split_name] = Dataset.from_list(
        build_publish_records(frame),
        features=Features(base_feature_spec),
    )

ds = DatasetDict(split_datasets)

# Validate the exact storage that will be pushed. Video must already be portable
# here: no /content paths, no repo-local scratch paths, and bytes present.
media_validation_ds = ds.cast_column("filmstrip", HFImage(decode=False))
for split_name, split_ds in media_validation_ds.items():
    for row in split_ds:
        assert row["video"]["bytes"] is not None, f"{split_name}: video bytes missing"
        assert row["video"]["path"] == row["video_filename"], row["video"]
        assert hashlib.sha256(row["video"]["bytes"]).hexdigest() == row["video_sha256"]
        assert row["filmstrip"]["bytes"] is not None, f"{split_name}: filmstrip bytes missing"
        assert row["filmstrip"]["path"] == row["filmstrip_filename"], row["filmstrip"]
        assert hashlib.sha256(row["filmstrip"]["bytes"]).hexdigest() == row["filmstrip_sha256"]
        if EXPORT_AUDIO_VARIANT:
            assert row["video_with_audio"]["bytes"] is not None, f"{split_name}: audio-bearing video bytes missing"
            assert row["video_with_audio"]["path"] == row["video_with_audio_filename"], row["video_with_audio"]
            assert hashlib.sha256(row["video_with_audio"]["bytes"]).hexdigest() == row["video_with_audio_sha256"]

print(ds)
preview = media_validation_ds["test"][0].copy()
for media_col in ["filmstrip", "video", "video_with_audio"]:
    if media_col in preview and isinstance(preview[media_col], dict):
        preview[media_col] = {
            "path": preview[media_col].get("path"),
            "bytes_len": len(preview[media_col]["bytes"]) if preview[media_col].get("bytes") is not None else None,
        }
print(preview)


DatasetDict({
    test: Dataset({
        features: ['species', 'valence', 'arousal', 'valence_arousal', 'video_filename', 'filmstrip_filename', 'video_sha256', 'filmstrip_sha256', 'duration_sec', 'source_url', 'source_article_url', 'source_doi', 'article_doi', 'license', 'video', 'filmstrip'],
        num_rows: 15
    })
})
{'species': 'Siberian tiger', 'valence': 'negative', 'arousal': 'high', 'valence_arousal': 'negative_high', 'video_filename': 'zoo_clip_0001.mp4', 'filmstrip_filename': 'zoo_clip_0001_filmstrip.jpg', 'video_sha256': '89637ec2ad6ba2279e94808c9ac415c82dcb07dda25247ffab720ad1724742f7', 'filmstrip_sha256': '6503beca76099244139fc78e177d8b64d8726af75e75b248d1902d913e26208c', 'duration_sec': 33.134, 'source_url': 'https://doi.org/10.6084/m9.figshare.c.7807931', 'source_article_url': 'https://www.nature.com/articles/s41598-025-28646-7', 'source_doi': '10.6084/m9.figshare.c.7807931', 'article_doi': '10.1038/s41598-025-28646-7', 'license': 'cc-by-4.0', 'video': {'path': 'zoo

In [10]:
#@title Dataset card scaffold
valence_counts = dict(Counter(public_df["valence"]))
arousal_counts = dict(Counter(public_df["arousal"]))
species_counts = dict(Counter(public_df["species"]))
active_audio_summary = "not run"
if "audio_audit_df" in globals():
    active_audio_summary = {
        "active_audio_sec_min": float(audio_audit_df["active_audio_sec"].min()),
        "active_audio_sec_median": float(audio_audit_df["active_audio_sec"].median()),
        "active_audio_sec_max": float(audio_audit_df["active_audio_sec"].max()),
        "active_audio_ratio_median": float(audio_audit_df["active_audio_ratio"].median()),
    }

card = f"""---
license: cc-by-4.0
language:
- en
pretty_name: OpenFARM Zoo Valence Arousal
size_categories:
- n<1K
tags:
- animal-welfare
- animal-behavior
- zoo-animals
- visual-affect
- video
- image
- valence
- arousal
- openfarm
---

# OpenFARM Zoo Valence Arousal

This prepared dataset derives from the Figshare collection **Emotion recognition study**, associated with Hiisivuori et al. (2025), **Human recognition of emotional valence and arousal of zoo animals**.

Source: {SOURCE_URL}
Article: {ARTICLE_URL}
Article DOI: {ARTICLE_DOI}
Source license: CC BY 4.0 according to the Figshare articles
Prepared: {datetime.now(timezone.utc).date().isoformat()}

## Scope

This is a tiny eval-only visual affect benchmark. Given a short zoo-animal visual stimulus, predict expert-labeled emotional valence and/or arousal.

- Species counts: {species_counts}
- Rows: {len(public_df)} eval examples
- Split: `test` only
- Valence labels: {valence_counts}
- Arousal labels: {arousal_counts}

These are expert emotional-state labels from the source study. They are not clinical pain scores or a complete welfare diagnosis.

## Prepared Media

The Figshare source provides three video compilations. Each row here is clipped from the compilation using black-frame separator detection and preserves the full stimulus segment used in the human study: real-speed clip, slow-motion clip, and still-frame portion. The primary `video` column is muted because the paper describes the study as visual recognition from mute clips.

Each row also includes a compact 3x3 `filmstrip` image for image-model evaluation.

Audio audit summary: {active_audio_summary}

Audio-bearing video exported in this dataset build: {bool(EXPORT_AUDIO_VARIANT)}. If present, `video_with_audio` should be treated as an explicit multimodal ablation, not the headline human-comparable benchmark.

## Data Shape

Main fields:

- `filmstrip`: HF Image feature with a 3x3 visual summary
- `video`: HF Video feature with the muted prepared MP4 stimulus
- `video_with_audio`: optional HF Video feature, present only when `EXPORT_AUDIO_VARIANT=True`
- `species`: target animal species
- `valence`: expert label, one of `negative`, `neutral`, `positive`
- `arousal`: expert label, one of `low`, `high`
- `valence_arousal`: combined helper label
- `duration_sec`: prepared stimulus duration
- `source_url`, `source_article_url`, `source_doi`, `article_doi`, `license`: source attribution fields

## Leakage Notes

Public rows use opaque media filenames and intentionally omit clip codes, source filenames, segment timestamps, expert interpretation notes, pre-formatted messages, task ids, and OpenFARM-specific row ids. The environment should build prompts from the media and selected non-target fields only.

## Sources

- Figshare collection: {SOURCE_URL}
- Article: {ARTICLE_URL}
- Article DOI: {ARTICLE_DOI}
"""
card_path = OUT_DIR / "README.md"
card_path.write_text(card, encoding="utf-8")
print(card)
print("wrote:", card_path)

---
license: cc-by-4.0
language:
- en
pretty_name: OpenFARM Zoo Valence Arousal
size_categories:
- n<1K
tags:
- animal-welfare
- animal-behavior
- zoo-animals
- visual-affect
- video
- image
- valence
- arousal
- openfarm
---

# OpenFARM Zoo Valence Arousal

This prepared dataset derives from the Figshare collection **Emotion recognition study**, associated with Hiisivuori et al. (2025), **Human recognition of emotional valence and arousal of zoo animals**.

Source: https://doi.org/10.6084/m9.figshare.c.7807931
Article: https://www.nature.com/articles/s41598-025-28646-7
Article DOI: 10.1038/s41598-025-28646-7
Source license: CC BY 4.0 according to the Figshare articles
Prepared: 2026-05-18

## Scope

This is a tiny eval-only visual affect benchmark. Given a short zoo-animal visual stimulus, predict expert-labeled emotional valence and/or arousal.

- Species counts: {'Siberian tiger': 5, 'Turkmenian markhor': 5, 'Barbary macaque': 5}
- Rows: 15 eval examples
- Split: `test` only
- Valen

In [12]:
#@title Optional push to Hugging Face Hub
# Keep PUSH_TO_HUB=False while iterating. This cell does not print tokens.

if PUSH_TO_HUB:
    if REQUIRE_FILMSTRIPS_FOR_PUSH:
        missing = [path for path in public_df["filmstrip"].map(Path) if not path.exists()]
        if missing:
            raise RuntimeError(f"Missing filmstrip files; first failures: {missing[:5]}")
    if REQUIRE_VIDEO_FOR_PUSH:
        missing = [path for path in public_df["video"].map(Path) if not path.exists()]
        if missing:
            raise RuntimeError(f"Missing video files; first failures: {missing[:5]}")
        if HFVideo is None:
            raise RuntimeError("datasets.Video is unavailable; update datasets before pushing video media.")
    if EXPORT_AUDIO_VARIANT:
        missing = [path for path in public_df["video_with_audio"].map(Path) if not path.exists()]
        if missing:
            raise RuntimeError(f"Missing audio-bearing video files; first failures: {missing[:5]}")

    # Refuse to push if the DatasetDict preview has not already materialized media bytes.
    push_check_ds = ds.cast_column("filmstrip", HFImage(decode=False))
    first_row = push_check_ds["test"][0]
    assert first_row["video"]["bytes"] is not None, "video bytes are missing from the DatasetDict preview"
    assert first_row["filmstrip"]["bytes"] is not None, "filmstrip bytes are missing from the DatasetDict preview"

    token = os.environ.get("HF_TOKEN")
    if token is None:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if token:
        login(token=token, add_to_git_credential=False)
    else:
        print("No HF_TOKEN found. You can run huggingface-cli login or set HF_TOKEN, then rerun this cell.")

    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=PRIVATE_REPO, exist_ok=True)
    ds.push_to_hub(HF_REPO_ID, private=PRIVATE_REPO, embed_external_files=True)
    api.upload_file(
        path_or_fileobj=str(card_path),
        path_in_repo="README.md",
        repo_id=HF_REPO_ID,
        repo_type="dataset",
    )

    # Reload from the Hub and verify that video bytes survived publication.
    remote = load_dataset(HF_REPO_ID, split="test", download_mode="force_redownload")
    remote = remote.cast_column("video", HFVideo(decode=False))
    remote = remote.cast_column("filmstrip", HFImage(decode=False))
    remote_row = remote[0]
    assert remote_row["video"]["bytes"] is not None, "published video bytes are missing"
    assert hashlib.sha256(remote_row["video"]["bytes"]).hexdigest() == remote_row["video_sha256"]
    assert remote_row["video"]["path"] == remote_row["video_filename"]
    assert remote_row["filmstrip"]["bytes"] is not None, "published filmstrip bytes are missing"
    assert hashlib.sha256(remote_row["filmstrip"]["bytes"]).hexdigest() == remote_row["filmstrip_sha256"]
    print("verified published media bytes:", {
        "video_path": remote_row["video"]["path"],
        "video_bytes": len(remote_row["video"]["bytes"]),
        "filmstrip_path": remote_row["filmstrip"]["path"],
        "filmstrip_bytes": len(remote_row["filmstrip"]["bytes"]),
    })
    print("pushed:", f"https://huggingface.co/datasets/{HF_REPO_ID}")
else:
    print("dry run only. Set PUSH_TO_HUB=True after reviewing outputs.")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/53.4M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

verified published media bytes: {'video_path': 'zoo_clip_0001.mp4', 'video_bytes': 2674567, 'filmstrip_path': 'zoo_clip_0001_filmstrip.jpg', 'filmstrip_bytes': 386176}
pushed: https://huggingface.co/datasets/oliveirabruno01/openfarm-zoo-valence-arousal


## Local Validation Snapshot

This notebook was refactored after inspecting the current prep notebooks (`catmeows`, `dog_pain`, `laying_hen_stress`, `ungulate`, plus older `aww_bench`, `soundwel`, and `sheep_apr`) for the project’s expected shape: `#@param` configuration, iterative source audits, visual diagnostics, DatasetDict preview, separate dataset-card scaffold, and separate optional Hub push.

Observed local output with `EXPORT_AUDIO_VARIANT=False`:
- rows: 15, split: `test` only
- species counts: Siberian tiger 5, Turkmenian markhor 5, Barbary macaque 5
- valence counts: negative 6, positive 6, neutral 3
- arousal counts: high 9, low 6
- duration range: 23.867s to 40.800s; total visual stimulus time about 8.27 minutes
- prepared media: 15 muted MP4 clips totaling about 46.8 MiB and 15 embedded 3x3 filmstrip JPEGs totaling about 4.2 MiB
- DatasetDict preview now materializes HF `Video`/`Image` media from bytes before push; `video.path` and `filmstrip.path` are opaque filenames, not local `/content` or repo-scratch paths
- source audio exists as AAC stereo, but median active audio ratio is about 0.48 because each stimulus also includes slow-motion/still/silent sections
- public schema check: no `openfarm_id`, no `task_id`, no `messages`, no clip codes, no source filenames, no segment timestamps, no expert interpretation notes, and opaque media filenames
